In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Load Dataset
df = pd.read_csv(r"D:\UPI Project\upi_fraud_dataset.csv")

# Feature Engineering
df["amount_deviation"] = abs(df["Amount"] - df["avg_amount"])

df["location_mismatch"] = (
    df["location"] != df["usual_location"]
).astype(int)

df["device_mismatch"] = (
    df["device"] != df["usual_device"]
).astype(int)

df["hour_deviation"] = abs(
    df["hour"] - df["usual_hour"]
)

# Selected Features
features = [
    "Amount",
    "avg_amount",
    "amount_deviation",
    "location_mismatch",
    "device_mismatch",
    "hour",
    "usual_hour",
    "hour_deviation"
]

X = df[features]
y = df["isFraud"]

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train Model
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

# Save Model
joblib.dump(model, "upi_fraud_model.pkl")

print("Model Saved Successfully!")

Model Saved Successfully!


In [2]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

# Load Model
model = joblib.load("upi_fraud_model.pkl")

st.set_page_config(
    page_title="UPI Fraud Detection",
    page_icon="💳"
)

st.title("💳 UPI Fraud Detection System")

# User Inputs
amount = st.number_input(
    "Transaction Amount",
    min_value=0.0,
    value=1000.0
)

avg_amount = st.number_input(
    "Average Amount",
    min_value=0.0,
    value=500.0
)

hour = st.slider(
    "Transaction Hour",
    0,
    23,
    12
)

usual_hour = st.slider(
    "Usual Transaction Hour",
    0,
    23,
    10
)

location_mismatch = st.selectbox(
    "Location Mismatch",
    [0, 1]
)

device_mismatch = st.selectbox(
    "Device Mismatch",
    [0, 1]
)

# Derived Features
amount_deviation = abs(amount - avg_amount)
hour_deviation = abs(hour - usual_hour)

# Prediction
if st.button("Predict Fraud"):

    input_data = pd.DataFrame({
        "Amount": [amount],
        "avg_amount": [avg_amount],
        "amount_deviation": [amount_deviation],
        "location_mismatch": [location_mismatch],
        "device_mismatch": [device_mismatch],
        "hour": [hour],
        "usual_hour": [usual_hour],
        "hour_deviation": [hour_deviation]
    })

    prediction = model.predict(input_data)[0]
    probability = model.predict_proba(input_data)[0][1]

    st.write(f"Fraud Probability: {probability*100:.2f}%")

    if prediction == 1:
        st.error("⚠ Fraudulent Transaction")
    else:
        st.success("✅ Legitimate Transaction")

Overwriting app.py


In [3]:
!pip install streamlit pandas scikit-learn joblib

In [ ]:
!python -m streamlit run app.py